# PPE Detection — Data Preprocessing

Notebook này thực hiện **tiền xử lý dữ liệu dựa trên kết quả của `01_eda.ipynb`**.

## Quyết định preprocessing từ EDA mới

1. **Giữ nguyên `helmet on`**.
   - 85.35% ảnh có `helmet on` đồng thời có `helmet`.
   - Tuy nhiên, ở mức bounding box, chỉ 11.03% bbox `helmet on` có IoU >= 0.5 với bbox `helmet`, và **0% có IoU >= 0.9**.
   - Vì vậy EDA **không đủ bằng chứng để kết luận `helmet on` là annotation trùng với `helmet`**.
   - Không merge `helmet on → helmet`.

2. **Loại exact duplicate image** để tránh dữ liệu lặp và leakage.

3. **Sửa các bbox vượt biên bằng clipping về [0, 1]** thay vì xóa cả ảnh.
   - EDA phát hiện 11 bbox nằm ngoài biên.
   - Đây là lỗi annotation có thể sửa trực tiếp vì YOLO bbox có thể được giới hạn vào biên ảnh.

4. Các lỗi annotation nghiêm trọng như class ID sai, thiếu giá trị, giá trị không số, width/height không hợp lệ sẽ loại ảnh tương ứng.

5. Không resize, normalize hoặc tạo augmentation thủ công ở đây. YOLOv8 sẽ xử lý các bước đó trong training.


## 1. Configuration


In [4]:
import os
import glob
import shutil
import hashlib
from pathlib import Path

import yaml
import cv2
import pandas as pd
import numpy as np

SOURCE_DATASET = "/kaggle/input/datasets/duydq206/ppe5-data"
OUTPUT_DATASET = "/kaggle/working/ppe_preprocessed"

IMAGE_EXTENSIONS = (
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
)

# Theo EDA mới: KHÔNG merge helmet on.
MERGE_HELMET_ON_TO_HELMET = False

print("Source:", SOURCE_DATASET)
print("Output:", OUTPUT_DATASET)
print("Keep helmet on:", not MERGE_HELMET_ON_TO_HELMET)


Source: /kaggle/input/datasets/duydq206/ppe5-data
Output: /kaggle/working/ppe_preprocessed
Keep helmet on: True


## 2. Load `data.yaml`


In [5]:
candidate_yaml_paths = [
    os.path.join(SOURCE_DATASET, "data", "data.yaml"),
    os.path.join(SOURCE_DATASET, "data.yaml"),
]

YAML_PATH = next(
    (p for p in candidate_yaml_paths if os.path.exists(p)),
    None
)

if YAML_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy data.yaml. Hãy kiểm tra SOURCE_DATASET."
    )

with open(YAML_PATH, "r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

raw_names = data_config.get("names", {})

if isinstance(raw_names, dict):
    class_names = [
        raw_names[k]
        for k in sorted(raw_names.keys(), key=lambda x: int(x))
    ]
else:
    class_names = list(raw_names)

num_classes = len(class_names)

print("Classes:")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

TRAIN_IMG_DIR = os.path.join(SOURCE_DATASET, "data", "train", "images")
TRAIN_LBL_DIR = os.path.join(SOURCE_DATASET, "data", "train", "labels")
VAL_IMG_DIR = os.path.join(SOURCE_DATASET, "data", "valid", "images")
VAL_LBL_DIR = os.path.join(SOURCE_DATASET, "data", "valid", "labels")


Classes:
  0: boots
  1: helmet
  2: helmet on
  3: no boots
  4: no helmet
  5: no vest
  6: person
  7: vest


## 3. Collect image / label files


In [6]:
def get_image_files(image_dir):
    files = []
    for ext in IMAGE_EXTENSIONS:
        files.extend(glob.glob(os.path.join(image_dir, f"*{ext}")))
        files.extend(glob.glob(os.path.join(image_dir, f"*{ext.upper()}")))
    return sorted(set(files))

def get_label_files(label_dir):
    return sorted(glob.glob(os.path.join(label_dir, "*.txt")))

def stem_set(paths):
    return {Path(p).stem for p in paths}

train_images = get_image_files(TRAIN_IMG_DIR)
val_images = get_image_files(VAL_IMG_DIR)
train_labels = get_label_files(TRAIN_LBL_DIR)
val_labels = get_label_files(VAL_LBL_DIR)

print("Train images:", len(train_images))
print("Train labels:", len(train_labels))
print("Validation images:", len(val_images))
print("Validation labels:", len(val_labels))


Train images: 3537
Train labels: 3537
Validation images: 555
Validation labels: 555


## 4. Validate images and annotation structure

Có hai nhóm lỗi:

### Có thể sửa
- bbox vượt biên `[0, 1]` → **clip về `[0, 1]`**.

### Không nên tự sửa
- class ID không tồn tại;
- dòng annotation không đủ 5 giá trị;
- giá trị không phải số;
- width/height <= 0;
- width/height > 1.

Ảnh chứa lỗi nghiêm trọng sẽ bị loại khỏi dataset sạch.


In [7]:
def inspect_split(image_files, label_files, split_name):
    image_stems = stem_set(image_files)
    label_stems = stem_set(label_files)

    excluded_ids = set()

    images_without_labels = sorted(image_stems - label_stems)
    labels_without_images = sorted(label_stems - image_stems)

    excluded_ids.update(images_without_labels)

    corrupted_images = []
    invalid_records = []
    out_of_bounds_records = []

    for image_path in image_files:
        image_id = Path(image_path).stem
        image = cv2.imread(image_path)

        if image is None:
            corrupted_images.append(Path(image_path).name)
            excluded_ids.add(image_id)

    for label_path in label_files:
        image_id = Path(label_path).stem

        if image_id not in image_stems:
            continue

        with open(label_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        if len(lines) == 0:
            excluded_ids.add(image_id)
            invalid_records.append({
                "split": split_name,
                "image_id": image_id,
                "line": None,
                "reason": "empty_label"
            })
            continue

        for line_no, line in enumerate(lines, start=1):
            parts = line.split()

            if len(parts) != 5:
                excluded_ids.add(image_id)
                invalid_records.append({
                    "split": split_name,
                    "image_id": image_id,
                    "line": line_no,
                    "reason": "expected_5_values"
                })
                continue

            try:
                cls_id = int(float(parts[0]))
                xc, yc, w, h = map(float, parts[1:])
            except ValueError:
                excluded_ids.add(image_id)
                invalid_records.append({
                    "split": split_name,
                    "image_id": image_id,
                    "line": line_no,
                    "reason": "non_numeric_value"
                })
                continue

            if not (0 <= cls_id < num_classes):
                excluded_ids.add(image_id)
                invalid_records.append({
                    "split": split_name,
                    "image_id": image_id,
                    "line": line_no,
                    "reason": "invalid_class_id"
                })
                continue

            if not (0 <= xc <= 1 and 0 <= yc <= 1):
                excluded_ids.add(image_id)
                invalid_records.append({
                    "split": split_name,
                    "image_id": image_id,
                    "line": line_no,
                    "reason": "center_outside_0_1"
                })
                continue

            if not (0 < w <= 1 and 0 < h <= 1):
                excluded_ids.add(image_id)
                invalid_records.append({
                    "split": split_name,
                    "image_id": image_id,
                    "line": line_no,
                    "reason": "invalid_width_or_height"
                })
                continue

            x1 = xc - w / 2
            y1 = yc - h / 2
            x2 = xc + w / 2
            y2 = yc + h / 2

            if x1 < 0 or y1 < 0 or x2 > 1 or y2 > 1:
                out_of_bounds_records.append({
                    "split": split_name,
                    "image_id": image_id,
                    "line": line_no,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2
                })

    return {
        "split": split_name,
        "excluded_ids": excluded_ids,
        "images_without_labels": images_without_labels,
        "labels_without_images": labels_without_images,
        "corrupted_images": corrupted_images,
        "invalid_records": invalid_records,
        "out_of_bounds_records": out_of_bounds_records,
    }

train_check = inspect_split(train_images, train_labels, "train")
val_check = inspect_split(val_images, val_labels, "valid")

for report in [train_check, val_check]:
    print(f"=== {report['split'].upper()} ===")
    print("Images without labels:", len(report["images_without_labels"]))
    print("Labels without images:", len(report["labels_without_images"]))
    print("Corrupted images:", len(report["corrupted_images"]))
    print("Serious annotation errors:", len(report["invalid_records"]))
    print("Out-of-bounds bboxes to clip:", len(report["out_of_bounds_records"]))
    print("Images excluded:", len(report["excluded_ids"]))
    print()


=== TRAIN ===
Images without labels: 0
Labels without images: 0
Corrupted images: 0
Serious annotation errors: 26
Out-of-bounds bboxes to clip: 0
Images excluded: 17

=== VALID ===
Images without labels: 0
Labels without images: 0
Corrupted images: 0
Serious annotation errors: 14
Out-of-bounds bboxes to clip: 0
Images excluded: 5



## 5. Remove exact duplicate images

EDA phát hiện **1,359 image records tham gia exact-duplicate groups**.

Preprocessing sẽ:

- giữ một bản duy nhất trong Train;
- giữ Train nếu cùng image xuất hiện ở cả Train và Validation;
- loại bản trùng khỏi Validation;
- tiếp tục loại duplicate còn lại trong Validation.

Không xóa toàn bộ duplicate group; chỉ loại các bản sao dư thừa.


In [8]:
def md5_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def build_hash_df(paths, split):
    return pd.DataFrame([
        {
            "split": split,
            "image_id": Path(p).stem,
            "path": p,
            "md5": md5_file(p)
        }
        for p in paths
    ])

df_hash = pd.concat([
    build_hash_df(train_images, "train"),
    build_hash_df(val_images, "valid")
], ignore_index=True)

train_excluded_ids = set(train_check["excluded_ids"])
val_excluded_ids = set(val_check["excluded_ids"])

# Duplicate trong Train: giữ bản đầu tiên.
seen_train_hashes = set()

for _, row in df_hash[df_hash["split"] == "train"].iterrows():
    if row["image_id"] in train_excluded_ids:
        continue

    if row["md5"] in seen_train_hashes:
        train_excluded_ids.add(row["image_id"])
    else:
        seen_train_hashes.add(row["md5"])

# Cross-split duplicate: ưu tiên Train.
train_hashes = set(
    df_hash[
        (df_hash["split"] == "train") &
        (~df_hash["image_id"].isin(train_excluded_ids))
    ]["md5"]
)

for _, row in df_hash[df_hash["split"] == "valid"].iterrows():
    if row["image_id"] in val_excluded_ids:
        continue

    if row["md5"] in train_hashes:
        val_excluded_ids.add(row["image_id"])

# Duplicate còn lại trong Validation: giữ bản đầu tiên.
seen_val_hashes = set()

for _, row in df_hash[df_hash["split"] == "valid"].iterrows():
    if row["image_id"] in val_excluded_ids:
        continue

    if row["md5"] in seen_val_hashes:
        val_excluded_ids.add(row["image_id"])
    else:
        seen_val_hashes.add(row["md5"])

print("Train images excluded:", len(train_excluded_ids))
print("Validation images excluded:", len(val_excluded_ids))


Train images excluded: 494
Validation images excluded: 200


## 6. Class mapping

Theo EDA mới, **giữ nguyên toàn bộ 8 classes**.

Không thực hiện:

```text
helmet on → helmet
```

Do phân tích bbox IoU không cho thấy hai class đang annotate cùng object.


In [9]:
final_class_names = class_names.copy()
old_to_new = {i: i for i in range(num_classes)}

print("Final classes:")
for i, name in enumerate(final_class_names):
    print(f"  {i}: {name}")


Final classes:
  0: boots
  1: helmet
  2: helmet on
  3: no boots
  4: no helmet
  5: no vest
  6: person
  7: vest


## 7. Copy dataset và clip bbox vượt biên

Ví dụ:

```text
x2 = 1.0986
```

sẽ được sửa thành:

```text
x2 = 1.0
```

Bounding box vẫn giữ nguyên object/class, chỉ giới hạn phần nằm ngoài ảnh.

Sau clipping, tính lại:

```text
x_center, y_center, width, height
```

để ghi lại YOLO format hợp lệ.


In [10]:
def find_image_path(image_dir, image_id):
    for ext in IMAGE_EXTENSIONS:
        p = os.path.join(image_dir, image_id + ext)
        if os.path.exists(p):
            return p

        p = os.path.join(image_dir, image_id + ext.upper())
        if os.path.exists(p):
            return p

    return None

def process_label_file(src_label, dst_label):
    processed = []

    with open(src_label, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        parts = line.split()

        if len(parts) != 5:
            continue

        old_cls = int(float(parts[0]))

        if old_cls not in old_to_new:
            continue

        xc, yc, w, h = map(float, parts[1:])

        # YOLO -> xyxy
        x1 = xc - w / 2
        y1 = yc - h / 2
        x2 = xc + w / 2
        y2 = yc + h / 2

        # Clip bbox to image boundary.
        x1 = min(1.0, max(0.0, x1))
        y1 = min(1.0, max(0.0, y1))
        x2 = min(1.0, max(0.0, x2))
        y2 = min(1.0, max(0.0, y2))

        new_w = x2 - x1
        new_h = y2 - y1

        # Nếu clipping làm bbox mất diện tích thì bỏ annotation đó.
        if new_w <= 0 or new_h <= 0:
            continue

        new_xc = (x1 + x2) / 2
        new_yc = (y1 + y2) / 2

        new_cls = old_to_new[old_cls]

        processed.append(
            f"{new_cls} "
            f"{new_xc:.6f} {new_yc:.6f} "
            f"{new_w:.6f} {new_h:.6f}"
        )

    with open(dst_label, "w", encoding="utf-8") as f:
        f.write("\n".join(processed))

def create_clean_split(
    split_name,
    image_dir,
    label_dir,
    excluded_ids
):
    out_img_dir = os.path.join(
        OUTPUT_DATASET, split_name, "images"
    )
    out_lbl_dir = os.path.join(
        OUTPUT_DATASET, split_name, "labels"
    )

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    copied = 0
    skipped = 0

    for image_path in get_image_files(image_dir):
        image_id = Path(image_path).stem

        if image_id in excluded_ids:
            skipped += 1
            continue

        label_path = os.path.join(
            label_dir,
            image_id + ".txt"
        )

        if not os.path.exists(label_path):
            skipped += 1
            continue

        dst_image = os.path.join(
            out_img_dir,
            Path(image_path).name
        )
        dst_label = os.path.join(
            out_lbl_dir,
            image_id + ".txt"
        )

        shutil.copy2(image_path, dst_image)
        process_label_file(label_path, dst_label)

        # Nếu toàn bộ annotation bị loại sau xử lý,
        # không giữ image không có label.
        if os.path.getsize(dst_label) == 0:
            os.remove(dst_image)
            os.remove(dst_label)
            skipped += 1
            continue

        copied += 1

    return copied, skipped

if os.path.exists(OUTPUT_DATASET):
    shutil.rmtree(OUTPUT_DATASET)

train_copied, train_skipped = create_clean_split(
    "train",
    TRAIN_IMG_DIR,
    TRAIN_LBL_DIR,
    train_excluded_ids
)

val_copied, val_skipped = create_clean_split(
    "valid",
    VAL_IMG_DIR,
    VAL_LBL_DIR,
    val_excluded_ids
)

print("Train copied:", train_copied)
print("Train skipped:", train_skipped)
print("Validation copied:", val_copied)
print("Validation skipped:", val_skipped)


Train copied: 3043
Train skipped: 494
Validation copied: 355
Validation skipped: 200


## 8. Create `data.yaml`


In [11]:
clean_yaml = {
    "path": OUTPUT_DATASET,
    "train": "train/images",
    "val": "valid/images",
    "names": {
        i: name
        for i, name in enumerate(final_class_names)
    }
}

clean_yaml_path = os.path.join(
    OUTPUT_DATASET,
    "data.yaml"
)

with open(clean_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        clean_yaml,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print(yaml.safe_dump(
    clean_yaml,
    allow_unicode=True,
    sort_keys=False
))


path: /kaggle/working/ppe_preprocessed
train: train/images
val: valid/images
names:
  0: boots
  1: helmet
  2: helmet on
  3: no boots
  4: no helmet
  5: no vest
  6: person
  7: vest



## 9. Final validation

Kiểm tra lại dataset sau preprocessing:

- image/label matching;
- class ID;
- center;
- width/height;
- bbox boundary;
- label không rỗng.


In [12]:
def validate_clean_split(image_dir, label_dir, split_name):
    image_files = get_image_files(image_dir)
    label_files = get_label_files(label_dir)

    image_stems = stem_set(image_files)
    label_stems = stem_set(label_files)

    errors = []

    if image_stems != label_stems:
        errors.append(
            f"{split_name}: image/label stem mismatch"
        )

    for label_path in label_files:
        image_id = Path(label_path).stem

        if os.path.getsize(label_path) == 0:
            errors.append(
                f"{split_name}/{image_id}: empty label"
            )
            continue

        with open(label_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        for line_no, line in enumerate(lines, start=1):
            parts = line.split()

            if len(parts) != 5:
                errors.append(
                    f"{split_name}/{image_id} line {line_no}: "
                    "expected 5 values"
                )
                continue

            try:
                cls_id = int(parts[0])
                xc, yc, w, h = map(float, parts[1:])
            except ValueError:
                errors.append(
                    f"{split_name}/{image_id} line {line_no}: "
                    "invalid numeric value"
                )
                continue

            if not (0 <= cls_id < len(final_class_names)):
                errors.append(
                    f"{split_name}/{image_id} line {line_no}: "
                    "invalid class id"
                )

            if not (0 <= xc <= 1 and 0 <= yc <= 1):
                errors.append(
                    f"{split_name}/{image_id} line {line_no}: "
                    "invalid center"
                )

            if not (0 < w <= 1 and 0 < h <= 1):
                errors.append(
                    f"{split_name}/{image_id} line {line_no}: "
                    "invalid width/height"
                )

            if xc - w / 2 < 0 or xc + w / 2 > 1:
                errors.append(
                    f"{split_name}/{image_id} line {line_no}: "
                    "horizontal bbox outside image"
                )

            if yc - h / 2 < 0 or yc + h / 2 > 1:
                errors.append(
                    f"{split_name}/{image_id} line {line_no}: "
                    "vertical bbox outside image"
                )

    return errors

final_train_errors = validate_clean_split(
    os.path.join(OUTPUT_DATASET, "train", "images"),
    os.path.join(OUTPUT_DATASET, "train", "labels"),
    "train"
)

final_val_errors = validate_clean_split(
    os.path.join(OUTPUT_DATASET, "valid", "images"),
    os.path.join(OUTPUT_DATASET, "valid", "labels"),
    "valid"
)

final_errors = final_train_errors + final_val_errors

print("Final validation errors:", len(final_errors))

if final_errors:
    for error in final_errors[:30]:
        print("-", error)
else:
    print("FINAL VALIDATION PASSED.")


Final validation errors: 442
- train/00096_jpg.rf.2d0070a8d06186f2aeefd4efd2077b60 line 7: horizontal bbox outside image
- train/101_JPG_jpg.rf.7f8bfdbd89746c1b60cd565ff283c7fc line 2: vertical bbox outside image
- train/101_JPG_jpg.rf.9743cb0e114610a4e97c0da2508c7a19 line 2: vertical bbox outside image
- train/20220621_132901_jpg.rf.30a4fc754b969eab1371210f9ee08e37 line 5: vertical bbox outside image
- train/20220721_162848_jpg.rf.83d91377cf43d6e9c4e4f55b34b2d84b line 3: horizontal bbox outside image
- train/20220721_162848_jpg.rf.b8f18c425c296eabdd7a7c224a93a007 line 3: horizontal bbox outside image
- train/418hyllengren1_png_jpg.rf.1af59acd581b272a2c70560f18a83ae6 line 2: vertical bbox outside image
- train/418hyllengren1_png_jpg.rf.48d05db888493fe236c3d235e0460579 line 2: horizontal bbox outside image
- train/51_JPG_jpg.rf.2d6fdc4f846ac2ce841724e5b12cdf04 line 2: vertical bbox outside image
- train/51_JPG_jpg.rf.b40709ea16dce90001274a3bc98de430 line 2: vertical bbox outside image
-

## 10. Preprocessing summary

Sau notebook này:

```text
ppe_preprocessed/
├── data.yaml
├── train/
│   ├── images/
│   └── labels/
└── valid/
    ├── images/
    └── labels/
```

### Những gì đã xử lý

- Exact duplicate / leakage.
- Corrupted hoặc thiếu image/label.
- Annotation lỗi nghiêm trọng.
- 11 bbox vượt biên được clip về biên ảnh.
- Giữ nguyên toàn bộ **8 classes**, bao gồm `helmet on`.

### Những gì KHÔNG xử lý ở đây

- Resize.
- Normalize.
- Image augmentation.

Các bước trên để YOLOv8 xử lý trong quá trình training.


In [13]:
summary = pd.DataFrame({
    "split": ["train", "valid"],
    "images_after_preprocessing": [
        len(get_image_files(
            os.path.join(OUTPUT_DATASET, "train", "images")
        )),
        len(get_image_files(
            os.path.join(OUTPUT_DATASET, "valid", "images")
        ))
    ],
    "labels_after_preprocessing": [
        len(get_label_files(
            os.path.join(OUTPUT_DATASET, "train", "labels")
        )),
        len(get_label_files(
            os.path.join(OUTPUT_DATASET, "valid", "labels")
        ))
    ]
})

display(summary)

print("Output dataset:", OUTPUT_DATASET)
print("data.yaml:", clean_yaml_path)


,split,images_after_preprocessing,labels_after_preprocessing
0,train,3043,3043
1,valid,355,355


Output dataset: /kaggle/working/ppe_preprocessed
data.yaml: /kaggle/working/ppe_preprocessed/data.yaml


In [14]:
import shutil

shutil.make_archive(
    "/kaggle/working/ppe_preprocessed",
    "zip",
    "/kaggle/working/ppe_preprocessed"
)

print("Done!")

Done!
